# Import all libraries needed

In [9]:
from pytrials.client import ClinicalTrials
import pandas as pd
import calendar
import time

# Loading to db
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv 
# loading variables from .env file
load_dotenv() 
import psycopg2
from sqlalchemy import create_engine

In [10]:
# max_studies limit is 1000
ct = ClinicalTrials()
#results = ct.get_full_studies(search_expr="endometriosis", max_studies = 50)

# Brainstorm pull
Querying the date from 1870 to 1963, it seems like the first record or article in Clinical Trials is in 1900.
Specifically for my keywords the first instance is in 1948.

--> FIRST BLOCK: 1948 - 1998 (QUERY YEAR)
--> SECOND BLOCK 1999 - 2007 (QUERY MONTH)
--> THIRD BLOCK 2008 - 2026 (QUERY MONTH, but loop through keywords)

# Test

In [ ]:
search_terms1 = "(Abdominal cerclage OR Abnormal pap smears OR Abnormal uterine bleeding OR Alzheimer's disease OR Amenorrhea OR Amniocentesis OR Arthritis OR Asthma OR Bacterial vaginosis OR Bleeding disorders OR Breast cancer OR Cancers OR Celiac disease OR Cervical cancer OR Cervical insufficiency OR Cervicitis OR Cesarean section (C-section) OR Chorionic villus sampling OR Colposcopy OR Congenital abnormalities OR Continence OR Contraception (birth control) OR Cystoscopy OR Depression and anxiety OR Dilation and curettage (D and C) OR Dysmenorrhea OR Endometrial ablation OR Endometrial biopsy OR Endometrial cancer OR Endometriosis OR Epilepsy OR Family planning OR Female genital cutting OR Fertility preservation OR Fibroma OR Gallstone disease OR Genetic conditions OR Gentle cesarean section OR Gestational diabetes OR Gestational trophoblastic disease OR Graves' disease OR Gynecologic cancer OR Gynecologic specialty care OR Gynecologic well-woman care OR Gynecological surgery OR Heart disease OR Heavy menstrual cycles OR High-risk pregnancy OR HIV OR HPV OR In-vitro fertilization OR Infertility OR Integrative medicine OR Interstitial cystitis OR Irritable bowel syndrome (IBS) OR Laparoscopic hysterectomy OR Laparoscopy OR Loop electrosurgical excision procedure (LEEP) OR Lupus OR Menopause OR Menorrhagia OR Menstrual conditions OR Miscarriage OR Morning sickness OR Multiple sclerosis OR Myomectomy OR Osteoarthritis OR Osteoporosis OR Ovarian cancer OR Ovarian cysts OR Ovarian fibroma OR Overactive bladder OR Pancreatic cystic neoplasm OR Pap test OR Pelvic inflammatory disease (PID) OR Pelvic organ prolapse OR Pelvic pain OR Pelvic ultrasound OR Perimenopause OR Placenta accreta OR Placenta previa OR Polycystic ovary syndrome (PCOS) OR Postpartum depression OR Preconception planning OR Pregnancy OR Pregnancy complications OR Pregnancy, first trimester OR Pregnancy, multiples OR Pregnancy, second trimester OR Pregnancy, third trimester OR Premenstrual dysphoric disorder (PMDD) OR Premenstrual syndrome (PMS) OR Prenatal care OR Prenatal genetic screening OR Preterm birth OR Primary care for women OR Recurrent pregnancy loss OR Rheumatoid arthritis OR Robotic hysterectomy OR Robotic myomectomy OR Sexually transmitted diseases (STDs) OR Sleep disorders OR Small vessel disease OR Sports injuries OR STIs OR Stress urinary incontinence OR Stroke OR Thyroid disease OR Transgender care OR Tubal ligation OR Turner syndrome OR Type I autoimmune hepatitis OR Ultrasound OR Urinary incontinence OR Urinary tract infections (UTIs) OR Uterine and vaginal prolapse OR Uterine fibroids OR Vaginal birth after cesarean section OR Vaginal cancer OR Vaginitis OR Vulvar cancer OR Vulvar dysplasia OR Vulvitis OR Yeast infection)"
fields1 = ct.get_study_fields(
    search_expr=f"{search_terms1} AND AREA[StartDate]RANGE[1948-01-01, 1998-12-31]",
    fields = ["NCT Number", 
              "Study Title", 
              "Study Status",
              "Brief Summary",
              "Conditions",
              "Primary Outcome Measures",
              "Sponsor",
              "Collaborators",
              "Sex",
              "Age",
              "Enrollment",
              "Study Type",
              "Funder Type",
              "Start Date",
              "Completion Date"],
    max_studies = 1000,
    fmt = "csv"
)

# convert to dataframe
df1 = pd.DataFrame.from_records(fields1[1:], columns = fields1[0])
#print(df1.info())
print(df1.info())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   NCT Number                1000 non-null   str  
 1   Study Title               1000 non-null   str  
 2   Study Status              1000 non-null   str  
 3   Brief Summary             1000 non-null   str  
 4   Conditions                1000 non-null   str  
 5   Primary Outcome Measures  1000 non-null   str  
 6   Sponsor                   1000 non-null   str  
 7   Collaborators             1000 non-null   str  
 8   Sex                       1000 non-null   str  
 9   Age                       1000 non-null   str  
 10  Enrollment                1000 non-null   str  
 11  Funder Type               1000 non-null   str  
 12  Study Type                1000 non-null   str  
 13  Start Date                1000 non-null   str  
 14  Completion Date           1000 non-null   str  
dtyp

# Find the first years for all keywords

In [12]:
### CHECK WHEN A MONTH WILL HIT 1000
# 1998 - 2027

search_terms = "(Abdominal cerclage OR Abnormal pap smears OR Abnormal uterine bleeding OR Alzheimer's disease OR Amenorrhea OR Amniocentesis OR Arthritis OR Asthma OR Bacterial vaginosis OR Bleeding disorders OR Breast cancer OR Cancers OR Celiac disease OR Cervical cancer OR Cervical insufficiency OR Cervicitis OR Cesarean section (C-section) OR Chorionic villus sampling OR Colposcopy OR Congenital abnormalities OR Continence OR Contraception (birth control) OR Cystoscopy OR Depression and anxiety OR Dilation and curettage (D and C) OR Dysmenorrhea OR Endometrial ablation OR Endometrial biopsy OR Endometrial cancer OR Endometriosis OR Epilepsy OR Family planning OR Female genital cutting OR Fertility preservation OR Fibroma OR Gallstone disease OR Genetic conditions OR Gentle cesarean section OR Gestational diabetes OR Gestational trophoblastic disease OR Graves' disease OR Gynecologic cancer OR Gynecologic specialty care OR Gynecologic well-woman care OR Gynecological surgery OR Heart disease OR Heavy menstrual cycles OR High-risk pregnancy OR HIV OR HPV OR In-vitro fertilization OR Infertility OR Integrative medicine OR Interstitial cystitis OR Irritable bowel syndrome (IBS) OR Laparoscopic hysterectomy OR Laparoscopy OR Loop electrosurgical excision procedure (LEEP) OR Lupus OR Menopause OR Menorrhagia OR Menstrual conditions OR Miscarriage OR Morning sickness OR Multiple sclerosis OR Myomectomy OR Osteoarthritis OR Osteoporosis OR Ovarian cancer OR Ovarian cysts OR Ovarian fibroma OR Overactive bladder OR Pancreatic cystic neoplasm OR Pap test OR Pelvic inflammatory disease (PID) OR Pelvic organ prolapse OR Pelvic pain OR Pelvic ultrasound OR Perimenopause OR Placenta accreta OR Placenta previa OR Polycystic ovary syndrome (PCOS) OR Postpartum depression OR Preconception planning OR Pregnancy OR Pregnancy complications OR Pregnancy, first trimester OR Pregnancy, multiples OR Pregnancy, second trimester OR Pregnancy, third trimester OR Premenstrual dysphoric disorder (PMDD) OR Premenstrual syndrome (PMS) OR Prenatal care OR Prenatal genetic screening OR Preterm birth OR Primary care for women OR Recurrent pregnancy loss OR Rheumatoid arthritis OR Robotic hysterectomy OR Robotic myomectomy OR Sexually transmitted diseases (STDs) OR Sleep disorders OR Small vessel disease OR Sports injuries OR STIs OR Stress urinary incontinence OR Stroke OR Thyroid disease OR Transgender care OR Tubal ligation OR Turner syndrome OR Type I autoimmune hepatitis OR Ultrasound OR Urinary incontinence OR Urinary tract infections (UTIs) OR Uterine and vaginal prolapse OR Uterine fibroids OR Vaginal birth after cesarean section OR Vaginal cancer OR Vaginitis OR Vulvar cancer OR Vulvar dysplasia OR Vulvitis OR Yeast infection)"

for year in range(1998, 2027):
    for month in range(1, 13):
        last_day = calendar.monthrange(year, month)[1]
        start_str = f"{year}-{month:02d}-01" # {month:02d} formats the month to MM because clinical trials is strict
        end_str = f"{year}-{month:02d}-{last_day:02d}"
        
        search_expr = f"{search_terms} AND AREA[StartDate]RANGE[{start_str}, {end_str}]"

        try: 
            data = ct.get_study_fields(
                search_expr=search_expr,
                fields=["NCT Number"],
                max_studies = 1000,
                fmt = "csv"
            )
            
            if data and len(data) > 0:
                studies_number = len(data) - 1
            else:
                studies_number = 0

            print(f"{year}-{month:02d}: Found {studies_number} studies")

        except Exception as e:
            print(f"Query error for {year}-{month:02d}: {e}")

        time.sleep(3)

# Found that 2007 is the first year that a month hit 1000 studies
        

1998-01: Found 101 studies
1998-02: Found 69 studies
1998-03: Found 82 studies
1998-04: Found 93 studies
1998-05: Found 72 studies
1998-06: Found 79 studies
1998-07: Found 88 studies
1998-08: Found 84 studies
1998-09: Found 110 studies
1998-10: Found 84 studies
1998-11: Found 68 studies
1998-12: Found 74 studies
1999-01: Found 140 studies
1999-02: Found 77 studies
1999-03: Found 106 studies
1999-04: Found 107 studies
1999-05: Found 82 studies
1999-06: Found 92 studies
1999-07: Found 119 studies
1999-08: Found 118 studies
1999-09: Found 148 studies
1999-10: Found 162 studies
1999-11: Found 103 studies
1999-12: Found 117 studies
2000-01: Found 291 studies
2000-02: Found 128 studies
2000-03: Found 133 studies
2000-04: Found 107 studies
2000-05: Found 142 studies
2000-06: Found 155 studies
2000-07: Found 119 studies
2000-08: Found 163 studies
2000-09: Found 181 studies
2000-10: Found 139 studies
2000-11: Found 114 studies
2000-12: Found 117 studies
2001-01: Found 239 studies
2001-02: Found

# Start looping and pulling data

## First year block: 1948 - 1998

In [ ]:
search_terms = "(Abdominal cerclage OR Abnormal pap smears OR Abnormal uterine bleeding OR Alzheimer's disease OR Amenorrhea OR Amniocentesis OR Arthritis OR Asthma OR Bacterial vaginosis OR Bleeding disorders OR Breast cancer OR Cancers OR Celiac disease OR Cervical cancer OR Cervical insufficiency OR Cervicitis OR Cesarean section (C-section) OR Chorionic villus sampling OR Colposcopy OR Congenital abnormalities OR Continence OR Contraception (birth control) OR Cystoscopy OR Depression and anxiety OR Dilation and curettage (D and C) OR Dysmenorrhea OR Endometrial ablation OR Endometrial biopsy OR Endometrial cancer OR Endometriosis OR Epilepsy OR Family planning OR Female genital cutting OR Fertility preservation OR Fibroma OR Gallstone disease OR Genetic conditions OR Gentle cesarean section OR Gestational diabetes OR Gestational trophoblastic disease OR Graves' disease OR Gynecologic cancer OR Gynecologic specialty care OR Gynecologic well-woman care OR Gynecological surgery OR Heart disease OR Heavy menstrual cycles OR High-risk pregnancy OR HIV OR HPV OR In-vitro fertilization OR Infertility OR Integrative medicine OR Interstitial cystitis OR Irritable bowel syndrome (IBS) OR Laparoscopic hysterectomy OR Laparoscopy OR Loop electrosurgical excision procedure (LEEP) OR Lupus OR Menopause OR Menorrhagia OR Menstrual conditions OR Miscarriage OR Morning sickness OR Multiple sclerosis OR Myomectomy OR Osteoarthritis OR Osteoporosis OR Ovarian cancer OR Ovarian cysts OR Ovarian fibroma OR Overactive bladder OR Pancreatic cystic neoplasm OR Pap test OR Pelvic inflammatory disease (PID) OR Pelvic organ prolapse OR Pelvic pain OR Pelvic ultrasound OR Perimenopause OR Placenta accreta OR Placenta previa OR Polycystic ovary syndrome (PCOS) OR Postpartum depression OR Preconception planning OR Pregnancy OR Pregnancy complications OR Pregnancy, first trimester OR Pregnancy, multiples OR Pregnancy, second trimester OR Pregnancy, third trimester OR Premenstrual dysphoric disorder (PMDD) OR Premenstrual syndrome (PMS) OR Prenatal care OR Prenatal genetic screening OR Preterm birth OR Primary care for women OR Recurrent pregnancy loss OR Rheumatoid arthritis OR Robotic hysterectomy OR Robotic myomectomy OR Sexually transmitted diseases (STDs) OR Sleep disorders OR Small vessel disease OR Sports injuries OR STIs OR Stress urinary incontinence OR Stroke OR Thyroid disease OR Transgender care OR Tubal ligation OR Turner syndrome OR Type I autoimmune hepatitis OR Ultrasound OR Urinary incontinence OR Urinary tract infections (UTIs) OR Uterine and vaginal prolapse OR Uterine fibroids OR Vaginal birth after cesarean section OR Vaginal cancer OR Vaginitis OR Vulvar cancer OR Vulvar dysplasia OR Vulvitis OR Yeast infection)"
target_fields =["NCT Number", "Study Title", "Study Status", "Brief Summary", "Conditions", "Primary Outcome Measures",
                "Sponsor", "Collaborators", "Sex", "Age", "Enrollment", "Study Type", "Funder Type", "Start Date", "Completion Date"]

dfs_list = []

for year in range(1948, 1999):
    start_str = f"{year}-01-01"
    end_str = f"{year}-12-31"
    year_query = f"{search_terms} AND AREA[StartDate]RANGE[{start_str}, {end_str}]"

    try:
        fields = ct.get_study_fields(search_expr=year_query, fields=target_fields,max_studies=1000,fmt="csv")
        
        if fields and len(fields) > 1:
            df = pd.DataFrame.from_records(fields[1:], columns=fields[0])
            dfs_list.append(df)

            studies_number = len(df)
            print(f"Year {year}: Found {studies_number} studies")

        else:
            print(f"Year {year}: No studies found.")

    except Exception as e:
        print(f"Error querying Year {year}: {e}")

    time.sleep(3)

if dfs_list:
    final_df = pd.concat(dfs_list, ignore_index = True)
else:
    print("No data collected")
    final_df = pd.DataFrame(columns=target_fields)
    

# convert to dataframe
#df = pd.DataFrame.from_records(fields[1:], columns = fields[0])
#print(df.info())

Year 1948: Found 1 studies
Year 1949: No studies found.
Year 1950: No studies found.
Year 1951: No studies found.
Year 1952: No studies found.
Year 1953: No studies found.
Year 1954: No studies found.
Year 1955: No studies found.
Year 1956: No studies found.
Year 1957: Found 1 studies
Year 1958: Found 1 studies
Year 1959: No studies found.
Year 1960: No studies found.
Year 1961: No studies found.
Year 1962: No studies found.
Year 1963: Found 1 studies
Year 1964: No studies found.
Year 1965: Found 3 studies
Year 1966: Found 3 studies
Year 1967: Found 1 studies
Year 1968: Found 1 studies
Year 1969: Found 2 studies
Year 1970: Found 1 studies
Year 1971: Found 7 studies
Year 1972: Found 6 studies
Year 1973: Found 6 studies
Year 1974: Found 2 studies
Year 1975: Found 4 studies
Year 1976: Found 14 studies
Year 1977: Found 22 studies
Year 1978: Found 18 studies
Year 1979: Found 12 studies
Year 1980: Found 17 studies
Year 1981: Found 11 studies
Year 1982: Found 12 studies
Year 1983: Found 31 st

In [15]:
print(final_df)

       NCT Number                                        Study Title  \
0     NCT00005121                             Framingham Heart Study   
1     NCT02833129  The Effects of Playing High School Football on...   
2     NCT00005122                               Evans County Studies   
3     NCT05537194  Effects of Dihydropyridine Calcium Channel Inh...   
4     NCT00005123                             Honolulu Heart Program   
...           ...                                                ...   
4078  NCT00390949  Scientific Evaluation of Peer Education and ST...   
4079  NCT00155038  Imaging Characteristics of Breast Cancer in Wo...   
4080  NCT00003139  Pilocarpine in Preventing Mucositis and Dry Mo...   
4081  NCT00003591  Radiation Therapy Plus Paclitaxel in Treating ...   
4082  NCT00003362  Vaccine Therapy Plus Immune Adjuvants in Treat...   

     Study Status                                      Brief Summary  \
0       COMPLETED  The Framingham Heart Study was initiated to 

# Second year block: 1999 - 2007

In [17]:
search_terms = "(Abdominal cerclage OR Abnormal pap smears OR Abnormal uterine bleeding OR Alzheimer's disease OR Amenorrhea OR Amniocentesis OR Arthritis OR Asthma OR Bacterial vaginosis OR Bleeding disorders OR Breast cancer OR Cancers OR Celiac disease OR Cervical cancer OR Cervical insufficiency OR Cervicitis OR Cesarean section (C-section) OR Chorionic villus sampling OR Colposcopy OR Congenital abnormalities OR Continence OR Contraception (birth control) OR Cystoscopy OR Depression and anxiety OR Dilation and curettage (D and C) OR Dysmenorrhea OR Endometrial ablation OR Endometrial biopsy OR Endometrial cancer OR Endometriosis OR Epilepsy OR Family planning OR Female genital cutting OR Fertility preservation OR Fibroma OR Gallstone disease OR Genetic conditions OR Gentle cesarean section OR Gestational diabetes OR Gestational trophoblastic disease OR Graves' disease OR Gynecologic cancer OR Gynecologic specialty care OR Gynecologic well-woman care OR Gynecological surgery OR Heart disease OR Heavy menstrual cycles OR High-risk pregnancy OR HIV OR HPV OR In-vitro fertilization OR Infertility OR Integrative medicine OR Interstitial cystitis OR Irritable bowel syndrome (IBS) OR Laparoscopic hysterectomy OR Laparoscopy OR Loop electrosurgical excision procedure (LEEP) OR Lupus OR Menopause OR Menorrhagia OR Menstrual conditions OR Miscarriage OR Morning sickness OR Multiple sclerosis OR Myomectomy OR Osteoarthritis OR Osteoporosis OR Ovarian cancer OR Ovarian cysts OR Ovarian fibroma OR Overactive bladder OR Pancreatic cystic neoplasm OR Pap test OR Pelvic inflammatory disease (PID) OR Pelvic organ prolapse OR Pelvic pain OR Pelvic ultrasound OR Perimenopause OR Placenta accreta OR Placenta previa OR Polycystic ovary syndrome (PCOS) OR Postpartum depression OR Preconception planning OR Pregnancy OR Pregnancy complications OR Pregnancy, first trimester OR Pregnancy, multiples OR Pregnancy, second trimester OR Pregnancy, third trimester OR Premenstrual dysphoric disorder (PMDD) OR Premenstrual syndrome (PMS) OR Prenatal care OR Prenatal genetic screening OR Preterm birth OR Primary care for women OR Recurrent pregnancy loss OR Rheumatoid arthritis OR Robotic hysterectomy OR Robotic myomectomy OR Sexually transmitted diseases (STDs) OR Sleep disorders OR Small vessel disease OR Sports injuries OR STIs OR Stress urinary incontinence OR Stroke OR Thyroid disease OR Transgender care OR Tubal ligation OR Turner syndrome OR Type I autoimmune hepatitis OR Ultrasound OR Urinary incontinence OR Urinary tract infections (UTIs) OR Uterine and vaginal prolapse OR Uterine fibroids OR Vaginal birth after cesarean section OR Vaginal cancer OR Vaginitis OR Vulvar cancer OR Vulvar dysplasia OR Vulvitis OR Yeast infection)"
target_fields =["NCT Number", "Study Title", "Study Status", "Brief Summary", "Conditions", "Primary Outcome Measures",
                "Sponsor", "Collaborators", "Sex", "Age", "Enrollment", "Study Type", "Funder Type", "Start Date", "Completion Date"]

dfs_list2 = []

for year in range(1999, 2008):
    for month in range(1, 13):
        last_day = calendar.monthrange(year, month)[1]
        start_str2 = f"{year}-{month:02d}-01" # {month:02d} formats the month to MM because clinical trials is strict
        end_str2 = f"{year}-{month:02d}-{last_day:02d}"
        
        month_query = f"{search_terms} AND AREA[StartDate]RANGE[{start_str2}, {end_str2}]"
        
        
        try:
            fields2 = ct.get_study_fields(search_expr=month_query, fields=target_fields, max_studies=1000, fmt="csv")
            
            if fields2 and len(fields2) > 1:
                df2 = pd.DataFrame.from_records(fields2[1:], columns=fields2[0])
                dfs_list2.append(df2)
                
                studies_number2 = len(df2)
                print(f"{year}-{month:02d}: Found {studies_number2} studies")
                
            else:
                print(f"{year}-{month:02d}: No studies found.")
                
        except Exception as e:
            print(f"Error querying Year {year}-{month:02d}: {e}")
            
        time.sleep(3)

if dfs_list2:
    final_df2 = pd.concat(dfs_list2, ignore_index = True)
else:
    print("No data collected")
    final_df2 = pd.DataFrame(columns=target_fields)

1999-01: Found 140 studies
1999-02: Found 77 studies
1999-03: Found 106 studies
1999-04: Found 107 studies
1999-05: Found 82 studies
1999-06: Found 92 studies
1999-07: Found 119 studies
1999-08: Found 118 studies
1999-09: Found 148 studies
1999-10: Found 162 studies
1999-11: Found 103 studies
1999-12: Found 117 studies
2000-01: Found 291 studies
2000-02: Found 128 studies
2000-03: Found 133 studies
2000-04: Found 107 studies
2000-05: Found 142 studies
2000-06: Found 155 studies
2000-07: Found 119 studies
2000-08: Found 163 studies
2000-09: Found 181 studies
2000-10: Found 139 studies
2000-11: Found 114 studies
2000-12: Found 117 studies
2001-01: Found 239 studies
2001-02: Found 153 studies
2001-03: Found 152 studies
2001-04: Found 189 studies
2001-05: Found 152 studies
2001-06: Found 158 studies
2001-07: Found 150 studies


KeyboardInterrupt: 

In [ ]:
print(final_df2)

        NCT Number                                        Study Title  \
0      NCT04308993  Percutaneous Endoscopic Biliary Exploration in...   
1      NCT00381511        Deferment of Imaging for Pulmonary Embolism   
2      NCT00115271  Antenatal Micronutrient Supplementation and Bi...   
3      NCT00003552  Chemotherapy and Peripheral Stem Cell Transpla...   
4      NCT03197233  Vitamin A and D Intake in Pregnancy, Infant Su...   
...            ...                                                ...   
37415  NCT01170091  Safety and Effect of Mirapex(Pramipexole) Tabl...   
37416  NCT00572442  Magnetocardiography (MCG) in Asymptomatic Indi...   
37417  NCT00782470  Evaluation of the Reasons and Consequences of ...   
37418  NCT03414502  Treatment of Rheumatoid Arthritis With DMARDs:...   
37419  NCT00592475  A Study to Assess the Safety and Effects of In...   

      Study Status                                      Brief Summary  \
0        COMPLETED  Patients with complex biliary 

## Third year block: 2008 - 2026

In [8]:
# Edited this to save each dataframe into a csv like I did for the paperscraper.ipynb
keywords = ["Abdominal cerclage", "Abnormal pap smears", "Abnormal uterine bleeding",  "Alzheimer's disease", "Amenorrhea",
            "Amniocentesis", "Arthritis", "Asthma", "Bacterial vaginosis", "Bleeding disorders", "Breast cancer", "Cancers",
            "Celiac disease", "Cervical cancer", "Cervical insufficiency", "Cervicitis", "Cesarean section (C-section)",
            "Chorionic villus sampling", "Colposcopy", "Congenital abnormalities", "Continence", "Contraception (birth control)",
            "Cystoscopy", "Depression and anxiety", "Dilation and curettage (D and C)", "Dysmenorrhea",  "Endometrial ablation",  "Endometrial biopsy",
            "Endometrial cancer", "Endometriosis", "Epilepsy", "Family planning", "Female genital cutting", "Fertility preservation",
            "Fibroma", "Gallstone disease",  "Genetic conditions", "Gentle cesarean section", "Gestational diabetes", "Gestational trophoblastic disease",
            "Graves' disease", "Gynecologic cancer", "Gynecologic specialty care", "Gynecologic well-woman care", "Gynecological surgery",  "Heart disease",
             "Heavy menstrual cycles", "High-risk pregnancy","HIV", "HPV", "In-vitro fertilization", "Infertility", "Integrative medicine", "Interstitial cystitis",
             "Irritable bowel syndrome (IBS)", "Laparoscopic hysterectomy", "Laparoscopy", "Loop electrosurgical excision procedure (LEEP)", "Lupus",
             "Menopause", "Menorrhagia", "Menstrual conditions", "Miscarriage", "Morning sickness", "Multiple sclerosis", "Myomectomy", "Osteoarthritis", "Osteoporosis",
             "Ovarian cancer", "Ovarian cysts", "Ovarian fibroma", "Overactive bladder", "Pancreatic cystic neoplasm", "Pap test", "Pelvic inflammatory disease (PID)",
             "Pelvic organ prolapse", "Pelvic pain", "Pelvic ultrasound", "Perimenopause", "Placenta accreta", "Placenta previa", "Polycystic ovary syndrome (PCOS)",
             "Postpartum depression", "Preconception planning", "Pregnancy", "Pregnancy complications", "Pregnancy, first trimester", "Pregnancy, multiples",
             "Pregnancy, second trimester", "Pregnancy, third trimester", "Premenstrual dysphoric disorder (PMDD)", "Premenstrual syndrome (PMS)", "Prenatal care", "Prenatal genetic screening", "Preterm birth",
             "Primary care for women", "Recurrent pregnancy loss", "Rheumatoid arthritis", "Robotic hysterectomy", "Robotic myomectomy", "Sexually transmitted diseases (STDs)", 
             "Sleep disorders", "Small vessel disease", "Sports injuries", "STIs", "Stress urinary incontinence", "Stroke", "Thyroid disease", "Transgender care", "Tubal ligation",
             "Turner syndrome", "Type I autoimmune hepatitis", "Ultrasound", "Urinary incontinence", "Urinary tract infections (UTIs)", "Uterine and vaginal prolapse", "Uterine fibroids",
             "Vaginal birth after cesarean section", "Vaginal cancer", "Vaginitis", "Vulvar cancer", "Vulvar dysplasia", "Vulvitis", "Yeast infection"
]

target_fields =["NCT Number", "Study Title", "Study Status", "Brief Summary", "Conditions", "Primary Outcome Measures",
                "Sponsor", "Collaborators", "Sex", "Age", "Enrollment", "Study Type", "Funder Type", "Start Date", "Completion Date"]

output_data = "raw_clincial_trials_2016_2026.csv"

for year in range(2016, 2027):
    for month in range(1, 13):
        last_day = calendar.monthrange(year, month)[1]
        start_str3 = f"{year}-{month:02d}-01" 
        end_str3 = f"{year}-{month:02d}-{last_day:02d}"

        # Look through keywords list
        for keyword in keywords:
            keyword_month_query = f"{keyword} AND AREA[StartDate]RANGE[{start_str3}, {end_str3}]"

            try:
                fields3 = ct.get_study_fields(search_expr=keyword_month_query, fields=target_fields, max_studies=1000, fmt="csv")

                if fields3 and len(fields3) > 1:
                    df3 = pd.DataFrame.from_records(fields3[1:], columns = fields3[0])
                    

                    file_name = os.path.isfile(output_data)
                    df3.to_csv(output_data, mode='a', header=not file_name, index=False)
                    print(f"Saved papers for '{keyword}' for this year {start_str3} - {end_str3}: {len(df3)}")
                    del df3  # Free up memory after saving to CSV

            except Exception as e:
                print(f" Error for keyword '{keyword}' in {year}-{month:02d}: {e}")

            time.sleep(2)

            


Saved papers for 'Abnormal pap smears' for this year 2016-01-01 - 2016-01-31: 2
Saved papers for 'Abnormal uterine bleeding' for this year 2016-01-01 - 2016-01-31: 3
Saved papers for 'Alzheimer's disease' for this year 2016-01-01 - 2016-01-31: 17
Saved papers for 'Amenorrhea' for this year 2016-01-01 - 2016-01-31: 1
Saved papers for 'Amniocentesis' for this year 2016-01-01 - 2016-01-31: 1
Saved papers for 'Arthritis' for this year 2016-01-01 - 2016-01-31: 68
Saved papers for 'Asthma' for this year 2016-01-01 - 2016-01-31: 33
Saved papers for 'Bacterial vaginosis' for this year 2016-01-01 - 2016-01-31: 2
Saved papers for 'Bleeding disorders' for this year 2016-01-01 - 2016-01-31: 166
Saved papers for 'Breast cancer' for this year 2016-01-01 - 2016-01-31: 90
Saved papers for 'Cancers' for this year 2016-01-01 - 2016-01-31: 696
Saved papers for 'Celiac disease' for this year 2016-01-01 - 2016-01-31: 9
Saved papers for 'Cervical cancer' for this year 2016-01-01 - 2016-01-31: 76
Saved paper

In [ ]:
# Save the data because computer crashed, continue from the year it left of at and concat the dataframes together
#if 'dfs_list3' in locals() and dfs_list3:
    #df = pd.concat(dfs_list3, ignore_index = True)

# Data Combination/Data Cleaning

In [ ]:
# This will be the final clinicaltrials dataframe with all of the three time blocks together
clinical_trials_data = pd.concat([final_df, final_df2, df], ignore_index = True)

In [ ]:
# First smaller clinical trials dataset that did not include all of the years
clinical_trials_data.to_csv("clinical_trials_data.csv", index = False)

In [ ]:
# concat all of the clinical trials dataframes together
# raw_clinical_trials
# raw_clinical_trials_2016_2026

In [33]:
df1 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/data/raw/raw_clinical_trials.csv")
df2 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/raw_clincial_trials_2016_2026.csv")

In [34]:
final_clinical_trials_data = pd.concat([df1, df2], ignore_index=True)

In [35]:
print(final_clinical_trials_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 739341 entries, 0 to 739340
Data columns (total 15 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   NCT Number                739341 non-null  str    
 1   Study Title               739341 non-null  str    
 2   Study Status              739341 non-null  str    
 3   Brief Summary             739341 non-null  str    
 4   Conditions                739338 non-null  str    
 5   Primary Outcome Measures  728511 non-null  str    
 6   Sponsor                   739341 non-null  str    
 7   Collaborators             257070 non-null  str    
 8   Sex                       739065 non-null  str    
 9   Age                       739341 non-null  str    
 10  Enrollment                736080 non-null  float64
 11  Funder Type               739341 non-null  str    
 12  Study Type                739341 non-null  str    
 13  Start Date                739341 non-null  str    
 14 

In [36]:
# Rename columns
final_clinical_trials_data = final_clinical_trials_data.rename(columns={'NCT Number': 'nct_number',
                              'Study Title': 'title',
                              'Study Status': 'study_status',
                              'Brief Summary': 'summary',
                              'Conditions': 'conditions',
                              'Primary Outcome Measures': 'pom',
                              'Sponsor': 'sponsor',
                              'Collaborators': 'collaborators',
                              'Sex': 'sex',
                              'Age': 'age',
                              'Enrollment': 'enrollment',
                              'Funder Type': 'funder_type',
                              'Study Type': 'study_type',
                              'Start Date': 'start_date',
                              'Completion Date': 'completion_date'})

In [ ]:
#Remove duplicates
print(final_clinical_trials_data['nct_number'].duplicated().sum())
print(final_clinical_trials_data['nct_number'].value_counts())

nct_number
NCT02742597    36
NCT04017754    30
NCT02458963    30
NCT02456792    30
NCT02789800    30
               ..
NCT07654660     1
NCT05915234     1
NCT07623213     1
NCT04003480     1
NCT07262944     1
Name: count, Length: 373282, dtype: int64


In [38]:
# Check what the data looks like for a specific nct number that has been duplicated
final_clinical_trials_data[final_clinical_trials_data['nct_number'] == 'NCT02742597'].head(5)


,nct_number,title,study_status,summary,conditions,pom,sponsor,collaborators,sex,age,enrollment,funder_type,study_type,start_date,completion_date
226044,NCT02742597,Patient-Centred Innovations for Persons With M...,COMPLETED,The aim of Patient-Centred Innovations for Per...,Hypertension|Depression|Anxiety|Musculoskeleta...,Evaluation of Intervention Effectiveness - Cha...,London Health Sciences Centre Research Institu...,"Western University, Canada|Université de Sherb...",ALL,"ADULT, OLDER_ADULT",175.0,OTHER,INTERVENTIONAL,2016-01-12,2022-10-19
226080,NCT02742597,Patient-Centred Innovations for Persons With M...,COMPLETED,The aim of Patient-Centred Innovations for Per...,Hypertension|Depression|Anxiety|Musculoskeleta...,Evaluation of Intervention Effectiveness - Cha...,London Health Sciences Centre Research Institu...,"Western University, Canada|Université de Sherb...",ALL,"ADULT, OLDER_ADULT",175.0,OTHER,INTERVENTIONAL,2016-01-12,2022-10-19
226146,NCT02742597,Patient-Centred Innovations for Persons With M...,COMPLETED,The aim of Patient-Centred Innovations for Per...,Hypertension|Depression|Anxiety|Musculoskeleta...,Evaluation of Intervention Effectiveness - Cha...,London Health Sciences Centre Research Institu...,"Western University, Canada|Université de Sherb...",ALL,"ADULT, OLDER_ADULT",175.0,OTHER,INTERVENTIONAL,2016-01-12,2022-10-19
226599,NCT02742597,Patient-Centred Innovations for Persons With M...,COMPLETED,The aim of Patient-Centred Innovations for Per...,Hypertension|Depression|Anxiety|Musculoskeleta...,Evaluation of Intervention Effectiveness - Cha...,London Health Sciences Centre Research Institu...,"Western University, Canada|Université de Sherb...",ALL,"ADULT, OLDER_ADULT",175.0,OTHER,INTERVENTIONAL,2016-01-12,2022-10-19
227510,NCT02742597,Patient-Centred Innovations for Persons With M...,COMPLETED,The aim of Patient-Centred Innovations for Per...,Hypertension|Depression|Anxiety|Musculoskeleta...,Evaluation of Intervention Effectiveness - Cha...,London Health Sciences Centre Research Institu...,"Western University, Canada|Université de Sherb...",ALL,"ADULT, OLDER_ADULT",175.0,OTHER,INTERVENTIONAL,2016-01-12,2022-10-19


In [39]:
fctd = final_clinical_trials_data.drop_duplicates(subset = ['nct_number'], keep = 'first')
fctd.info()

<class 'pandas.DataFrame'>
Index: 373282 entries, 0 to 739335
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   nct_number       373282 non-null  str    
 1   title            373282 non-null  str    
 2   study_status     373282 non-null  str    
 3   summary          373282 non-null  str    
 4   conditions       373280 non-null  str    
 5   pom              363741 non-null  str    
 6   sponsor          373282 non-null  str    
 7   collaborators    125975 non-null  str    
 8   sex              373122 non-null  str    
 9   age              373282 non-null  str    
 10  enrollment       370212 non-null  float64
 11  funder_type      373282 non-null  str    
 12  study_type       373282 non-null  str    
 13  start_date       373282 non-null  str    
 14  completion_date  365288 non-null  str    
dtypes: float64(1), str(14)
memory usage: 612.2 MB


In [ ]:
# Make sure date is consistent
print(fctd['start_date'].astype(str).str.len().value_counts())
print(fctd['completion_date'].astype(str).str.len().value_counts())

# Change format for all dates to 10 not 7
fctd['start_date'] = pd.to_datetime(fctd['start_date'], format='mixed', errors='coerce')
fctd['completion_date'] = pd.to_datetime(fctd['completion_date'], format='mixed', errors='coerce')

start_date
10    240168
7     133114
Name: count, dtype: int64
completion_date
10.0    220868
7.0     144420
Name: count, dtype: int64


In [42]:
print(fctd['start_date'].astype(str).str.len().value_counts())
print(fctd['completion_date'].astype(str).str.len().value_counts())

start_date
10    373282
Name: count, dtype: int64
completion_date
10.0    365288
Name: count, dtype: int64


In [43]:
# Save final clean raw data to one csv file
fctd.to_csv("raw_clinical_trials_data.csv", index = False)

# Send raw data to database warehouse

In [44]:
# Connect to the database
conn = psycopg2.connect(
    dbname=os.getenv("DBNAME"),
    user=os.getenv("DBUSER"),
    password=os.getenv("DBPASSWORD"),
    port=os.getenv("DBPORT"),
    host=os.getenv("DBHOST")
)
conn_string=os.getenv("CONNSTRING")

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

In [47]:
# Load data to db
fctd.to_sql("final_raw_clinical_trials", engine, if_exists="append", index=False, method="multi", chunksize=300)

373282